# Paper 21 backbone on YOLO26n: SPDConv at the P3 down-sampling

| Part | Source |
|---|---|
| **Backbone (layers 0–10)** | **Paper 21:** the stride-2 `Conv` at layer 3 (P3 down-sampling) replaced by **SPDConv**; every other backbone layer is YOLO26n base |
| Neck FPN + PAN (layers 11–22) | YOLO26n base, unchanged (the WCAF neck of Paper 21 is not used) |
| Head (Detect, layer 23) | YOLO26n base, unchanged (one-to-one, NMS-free); Focaler-MPDIoU loss not used |

The baseline ("Standard YOLO26n Backbone") is **not trained here**; its values are already filled in the final table.

**Kaggle settings:** Accelerator = **GPU T4 x1**, Internet = **On**, keep the notebook **Private**

## 1. Setup

In [ ]:
# ultralytics version is pinned: use the same version on Kaggle, Raspberry Pi 5 and Jetson
!pip install -q ultralytics==8.4.137 roboflow

In [ ]:
import os, sys, gc, shutil, random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch, torchvision
import ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (turn on the GPU accelerator)")

WORK = Path("/kaggle/working")
os.chdir(WORK)
sys.path.insert(0, str(WORK))  # so spd_backbone.py can be imported

## 2. Dataset (download from Roboflow)

The Roboflow API key is written in the cell below. Keep this notebook **Private** on Kaggle.

In [ ]:
import os, yaml
from roboflow import Roboflow

os.chdir("/kaggle/working")

# Roboflow API key (keep this notebook Private on Kaggle)
API_KEY = "CxHgc1Nn3FK5vYNsBtJW"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("md-mahibullah").project("digitalaccess-2-copy")
dataset = project.version(4).download("yolo26")  # Roboflow YOLO26 export (same label format as YOLOv8)

# Rename and fix paths
new = "/kaggle/working/digitalaccess"
if os.path.exists(dataset.location) and not os.path.exists(new):
    os.rename(dataset.location, new)

# Update YAML
yaml_path = f"{new}/data.yaml"
with open(yaml_path) as f:
    data = yaml.safe_load(f)
data["path"], data["train"], data["val"] = new, "train/images", "valid/images"
if os.path.exists(f"{new}/test/images"):
    data["test"] = "test/images"
else:
    data.pop("test", None)
with open(yaml_path, "w") as f:
    yaml.dump(data, f)

# the following cells use these variables
DATA_ROOT = Path(new)
DATA_YAML = yaml_path
names = data["names"]
assert data.get("test"), "No test split: the evaluation and FPS steps need a test split"

for s in ["train", "valid", "test"]:
    print(f"{s:6s}: {len(list((DATA_ROOT / s / 'images').glob('*')))} images")
print("nc =", len(names))
print("✅ Dataset prepared!")

## 3. Paper 21 backbone: SPDConv

**What SPDConv does** (Paper 21, Section 2.3.1):
1. **Space-to-depth**: every 2 × 2 block of pixels is moved into the channels, so H and W are halved and the channels
   become 4×, *without dropping any pixel* (a stride-2 conv skips positions).
2. A **3×3 conv with stride 1** (BN + SiLU) maps the 4 × C channels to the output channels.

**Where it sits in YOLO26n.**

| Layer | Base YOLO26n | Paper 21 backbone | Output |
|---|---|---|---|
| 3 | Conv 3×3, stride 2 (64 → 64) | **SPDConv** (space-to-depth → Conv 3×3 s1, 256 → 64) | P3/8 (80 × 80) |

Input and output channels stay the same, so no layer index changes and the neck connections stay the same.

**Pretrained weights (exact warm start).** A 3×3 stride-2 conv is a special case of space-to-depth followed by a 3×3
stride-1 conv: each tap of the old kernel maps to one (sub-image, offset) position of the new kernel. So the new conv
is initialised from the pretrained layer-3 conv (and its BatchNorm is copied), and at the start SPDConv gives
**exactly the same output** as the base layer. The other new kernel positions start at zero and are learned during
training. The structure check below prints the difference (it should be about 1e-6).

**Cost.** The conv sees 4× more input channels, so layer 3 has about 4× more parameters and FLOPs
(Paper 21: 5.19 → 6.46 GFLOPs). The Raspberry Pi 5 FPS will show whether this is acceptable.

In [ ]:
%%writefile spd_backbone.py
"""Paper 21 backbone for YOLO26n: stride-2 Conv at layer 3 -> SPDConv (space-to-depth + 3x3 stride-1 Conv), exact warm start."""
import torch
import torch.nn as nn

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.modules.conv import Conv
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

SPD_LAYERS = (3,)

# space-to-depth sub-images in channel order: (row offset a, column offset b)
SUBS = ((0, 0), (1, 0), (0, 1), (1, 1))
# tap of a 3x3 stride-2 kernel (dy = -1, 0, +1) -> (sub-image offset, tap of the new 3x3 stride-1 kernel)
TAP = {0: (1, 0), 1: (0, 1), 2: (1, 1)}


class SPDConv(nn.Module):
    """Space-to-depth (2x2 -> channels) followed by a 3x3 stride-1 Conv (BN + SiLU)."""

    def __init__(self, c1, c2):
        super().__init__()
        self.conv = Conv(4 * c1, c2, 3, 1)

    def forward(self, x):
        x = torch.cat([x[..., a::2, b::2] for a, b in SUBS], 1)
        return self.conv(x)


@torch.no_grad()
def init_from_strided(spd, old):
    """Set the SPDConv weights so that it reproduces the pretrained 3x3 stride-2 Conv exactly; copy its BatchNorm."""
    c1 = old.conv.in_channels
    w_old = old.conv.weight
    w_new = torch.zeros_like(spd.conv.conv.weight)
    for ky, (a, ny) in TAP.items():
        for kx, (b, nx) in TAP.items():
            s = SUBS.index((a, b))
            w_new[:, s * c1:(s + 1) * c1, ny, nx] += w_old[:, :, ky, kx]
    spd.conv.conv.weight.copy_(w_new)
    spd.conv.bn.load_state_dict(old.bn.state_dict())


def add_spd(model, layers=SPD_LAYERS):
    """Replace the stride-2 Conv at the given backbone layers with SPDConv (exact warm start). Returns the model."""
    for i in layers:
        old = model.model[i]
        assert isinstance(old, Conv) and old.conv.kernel_size == (3, 3) and old.conv.stride == (2, 2), f"layer {i} is not a 3x3 s2 Conv"
        assert old.conv.padding == (1, 1) and old.conv.groups == 1 and old.conv.dilation == (1, 1)
        new = SPDConv(old.conv.in_channels, old.conv.out_channels).to(old.conv.weight.device)
        init_from_strided(new, old)
        new.i, new.f, new.type = old.i, old.f, "SPDConv"  # keep the routing attributes used by the model's forward
        new.np = sum(p.numel() for p in new.parameters())
        model.model[i] = new
    return model


class SPDTrainer(DetectionTrainer):
    """Loads the pretrained weights first, then swaps layer 3 for SPDConv initialised from the pretrained conv."""

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DetectionModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=False)
        if weights:
            model.load(weights)
        add_spd(model)
        if RANK in {-1, 0}:
            print(f"Paper 21 backbone: SPDConv at layers {SPD_LAYERS}")
        if verbose and RANK in {-1, 0}:
            model.info()
        return model


In [ ]:
import importlib, spd_backbone
importlib.reload(spd_backbone)
from spd_backbone import add_spd, SPDTrainer
from ultralytics.nn.tasks import DetectionModel, yaml_model_load
from ultralytics.utils.torch_utils import get_flops

def fused_params_gflops(model, imgsz=640):
    m = deepcopy(model).float().cpu().eval()
    m.fuse(verbose=False)
    return sum(p.numel() for p in m.parameters()) / 1e6, get_flops(m, imgsz)

# structure check (untrained): base vs Paper 21 backbone
base = DetectionModel(yaml_model_load("yolo26n.yaml"), nc=len(names), verbose=False).eval()
spd = add_spd(deepcopy(base)).eval()
with torch.no_grad():
    _ = spd(torch.zeros(1, 3, 640, 640))  # full forward check
    x = torch.randn(1, base.model[3].conv.in_channels, 160, 160)
    yb, ys = base.model[3](x), spd.model[3](x)
print("layer 3:", type(spd.model[3]).__name__, "| output", tuple(ys.shape[1:]))
print(f"warm start: max |base layer 3 - SPDConv| = {(yb - ys).abs().max():.2e} (should be about 1e-6)")
for tag, m in (("base", base), ("Paper 21 backbone", spd)):
    pm, gf = fused_params_gflops(m)
    print(f"{tag:18s} Params {pm:.4f} M | GFLOPs {gf:.3f}")

## 4. Training

Only the Paper 21 backbone model is trained.

In [ ]:
from ultralytics import YOLO

EPOCHS  = 100
IMGSZ   = 640
BATCH   = 16
WORKERS = 2
SEED    = 0
PROJECT = str(WORK / "runs")

COMMON = dict(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, deterministic=True,
    pretrained="yolo26n.pt", device=0, workers=WORKERS, cache=False, patience=30, cos_lr=False,
    project=PROJECT, exist_ok=True, plots=True,
    # channels_last=False: ultralytics' MuSGD crashes on channels_last weights ("view size is not compatible").
    optimizer="MuSGD", lr0=0.01, momentum=0.9, channels_last=False,
)

# (row label in the table, run name)
VARIANTS = [
    ("SPDConv Backbone (Paper 21)", "bb_spd"),
]

def best_path(run):
    return Path(PROJECT) / f"{run}_s{SEED}" / "weights" / "best.pt"

for label, run in VARIANTS:
    print(f"\n========== {label} ({run}) ==========")
    model = YOLO("yolo26n.yaml")
    model.train(trainer=SPDTrainer, **dict(COMMON, name=f"{run}_s{SEED}"))
    del model; gc.collect(); torch.cuda.empty_cache()

## 5. Test-set evaluation: Precision, Recall, F1-score, mAP@50, Params, GFLOPs

Params and GFLOPs are counted on the fused inference model, the same way as the baseline row.

In [ ]:
import spd_backbone  # must be imported before loading an SPDConv checkpoint

rows = []
for label, run in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, train it first"); continue
    r = YOLO(str(w)).val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=16, device=0, plots=False, verbose=False)
    P, R = float(r.box.mp), float(r.box.mr)
    params, gflops = fused_params_gflops(YOLO(str(w)).model, IMGSZ)
    rows.append({"run": run, "Backbone Variant": label,
                 "Params (M)": round(params, 2), "GFLOPs": round(gflops, 2),
                 "Precision": round(P, 3), "Recall": round(R, 3),
                 "F1-score": round(2 * P * R / (P + R + 1e-9), 3),
                 "mAP@50": round(float(r.box.map50), 3)})
METRICS = pd.DataFrame(rows)
METRICS.drop(columns="run")

## 6. Package for Raspberry Pi 5 and Jetson

Creates `paper21_backbone_edge_bench.zip` in the **Output** tab with:

| Item | Used on |
|---|---|
| `bb_spd_ncnn_model/` (NCNN model, imgsz 640) | Raspberry Pi 5 |
| `bb_spd.pt` | Jetson |
| `bench_images/` (50 real test images, seed 0) | both |
| `spd_backbone.py` | Jetson (needed to load the SPDConv `.pt`) |
| `pi5_bench.py`, `jetson_bench.py` | benchmark scripts |

Both scripts: batch 1, imgsz 640, cycle through the 50 real test images (loaded into memory first),
100 warm-up runs, then 1000 timed runs end-to-end (preprocessing + inference + postprocessing), and print the **median** ms and FPS.
Unzip into its own folder, e.g. `python -m zipfile -e paper21_backbone_edge_bench.zip paper21_backbone_edge_bench`.

In [ ]:
EDGE_IMGSZ = 640
N_BENCH_IMAGES = 50
EDGE = WORK / "paper21_backbone_edge_bench"
shutil.rmtree(EDGE, ignore_errors=True)
EDGE.mkdir()
shutil.copy("spd_backbone.py", EDGE / "spd_backbone.py")

# 50 real test images (fixed seed, so every model and device uses the same images)
test_imgs = sorted(p for p in (DATA_ROOT / "test" / "images").glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
random.Random(SEED).shuffle(test_imgs)
(EDGE / "bench_images").mkdir()
for p in test_imgs[:N_BENCH_IMAGES]:
    shutil.copy(p, EDGE / "bench_images" / p.name)
print(f"bench_images: {len(list((EDGE / 'bench_images').glob('*')))} real test images")

for _, run in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, skipped"); continue
    shutil.copy(w, EDGE / f"{run}.pt")
    p = YOLO(str(w)).export(format="ncnn", imgsz=EDGE_IMGSZ, half=False)
    shutil.copytree(p, EDGE / f"{run}_ncnn_model")  # ultralytics needs the "_ncnn_model" suffix
    print(f"{run:12s} -> {run}.pt, {run}_ncnn_model/")

COMMON_BENCH = """
import glob, statistics, sys, time
import cv2

IMGSZ, WARMUP, RUNS = {imgsz}, 100, 1000
imgs = [cv2.imread(p) for p in sorted(glob.glob("bench_images/*"))]
imgs = [im for im in imgs if im is not None]
if not imgs:
    sys.exit("No images found in bench_images/. Run this script inside the unzipped folder.")
print(f"{{len(imgs)}} real test images, imgsz {{IMGSZ}}, batch 1, {{WARMUP}} warm-up + {{RUNS}} timed runs (median)")


def bench(model, **kw):
    for i in range(WARMUP):
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
    times = []
    for i in range(RUNS):
        t = time.perf_counter()
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
        times.append((time.perf_counter() - t) * 1000)
    return statistics.median(times)
""".format(imgsz=EDGE_IMGSZ)

PI5 = '''# Raspberry Pi 5 benchmark (NCNN). Run inside the unzipped folder:  python pi5_bench.py
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*_ncnn_model"))
if not models:
    sys.exit("No *_ncnn_model folders found.")
for path in models:
    ms = bench(YOLO(path, task="detect"))
    print(f"{path.replace('_ncnn_model', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

JETSON = '''# Jetson benchmark (PyTorch, CUDA FP16). Run inside the unzipped folder:  python jetson_bench.py
import sys
import torch

if not torch.cuda.is_available():
    sys.exit("WARNING: CUDA GPU not available. Install the NVIDIA Jetson PyTorch wheel for your JetPack, then run again.")
print("GPU:", torch.cuda.get_device_name(0))

sys.path.insert(0, ".")
import spd_backbone  # needed to load the SPDConv checkpoint
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*.pt"))
if not models:
    sys.exit("No .pt files found.")
for path in models:
    ms = bench(YOLO(path, task="detect"), device=0, half=True)
    print(f"{path.replace('.pt', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

(EDGE / "pi5_bench.py").write_text(PI5)
(EDGE / "jetson_bench.py").write_text(JETSON)
shutil.make_archive(str(WORK / "paper21_backbone_edge_bench"), "zip", EDGE)
print("\nDownload paper21_backbone_edge_bench.zip from the Output tab.")

### How to run on the devices

**Raspberry Pi 5** (same Pi and same conditions as the baseline measurement)
1. Activate the Python environment that has `ultralytics==8.4.137` and `ncnn`.
2. `echo performance | sudo tee /sys/devices/system/cpu/cpu*/scaling_governor`
3. Go into the unzipped folder and run `python pi5_bench.py`.
4. Run it 3 times with a 2–3 minute pause; use the middle value. Check `vcgencmd measure_temp` and `vcgencmd get_throttled` (`throttled=0x0` = OK).

**Jetson**
1. NVIDIA Jetson PyTorch + torchvision for your JetPack, then `ultralytics==8.4.137`.
2. `sudo nvpmodel -m 0` (check with `sudo nvpmodel -q`) and `sudo jetson_clocks`.
3. Go into the unzipped folder and run `python jetson_bench.py`. Run 3 times; use the middle value.

## 7. Final backbone table

The baseline row is already filled in. Type the median FPS of the Paper 21 backbone model
(and the baseline Jetson FPS when you have it), keep `None` for a missing value (shown as `--`), then run the cell.
Run section 5 first in the same session (it fills `METRICS`).

In [ ]:
# baseline values (identical YOLO26n model)
BASE_ROW = {"Backbone Variant": "Standard YOLO26n Backbone", "Params (M)": 2.38, "GFLOPs": 5.34,
            "Precision": 0.971, "Recall": 0.937, "F1-score": 0.953, "mAP@50": 0.958,
            "FPS (Rasp-5)": 13.38, "FPS (Jetson)": None}

FPS_PI5    = {"bb_spd": None}
FPS_JETSON = {"bb_spd": None}

table = METRICS.copy()
table["FPS (Rasp-5)"] = table["run"].map(lambda r: FPS_PI5.get(r))
table["FPS (Jetson)"] = table["run"].map(lambda r: FPS_JETSON.get(r))
COLS = ["Backbone Variant", "Params (M)", "GFLOPs", "Precision", "Recall", "F1-score", "mAP@50", "FPS (Rasp-5)", "FPS (Jetson)"]
table = pd.concat([pd.DataFrame([BASE_ROW]), table[COLS]], ignore_index=True)[COLS]
table = table.astype(object).where(table.notna(), "--")
table.style.hide(axis="index")

## 8. Zip all results

Run this last. It creates **`paper21_backbone_all_results.zip`** in the **Output** tab with:

| Folder in the zip | Contents |
|---|---|
| `paper21_backbone_edge_bench/` | NCNN + `.pt` model, 50 bench images, `spd_backbone.py`, `pi5_bench.py`, `jetson_bench.py` |
| `weights/` | `best.pt`, renamed `bb_spd_best.pt` |
| `runs/` | full training folder: `best.pt`, `last.pt`, `results.csv`, training curves, confusion matrix, PR curves |

In [ ]:
import zipfile

ALL_ZIP = WORK / "paper21_backbone_all_results.zip"
folders = [WORK / "paper21_backbone_edge_bench", WORK / "runs"]

with zipfile.ZipFile(ALL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders:
        if not folder.exists():
            print(f"{folder.name}: not found, skipped"); continue
        n = 0
        for f in folder.rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(WORK))
                n += 1
        print(f"{folder.name + '/':30s} {n} files")
    for _, run in VARIANTS:
        w = best_path(run)
        if w.exists():
            z.write(w, f"weights/{run}_best.pt")
            print(f"weights/{run}_best.pt")

print(f"\nDownload {ALL_ZIP.name} ({ALL_ZIP.stat().st_size / 1e6:.1f} MB) from the Output tab.")